In [1]:
# === 매 세션 시작 시 이 셀 1번만 실행 ===
#Drive 마운트 + ultralytics 설치 + BASE 설정 + 커스텀 yaml 재생성

from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics -q

import os
BASE = "/content/drive/MyDrive/driving"
os.chdir(BASE)

# ByteTrack 커스텀 설정 파일 재생성
# (/content/ 폴더는 세션 끊기면 날아가므로 매번 다시 만들어야 함)
bytetrack_config = """tracker_type: bytetrack
track_high_thresh: 0.4
track_low_thresh: 0.1
new_track_thresh: 0.5
track_buffer: 90
match_thresh: 0.8
fuse_score: True
"""
with open('/content/custom_bytetrack.yaml', 'w') as f:
    f.write(bytetrack_config)

import torch
print("GPU:", torch.cuda.is_available())
print("BASE:", BASE)
print("준비 완료")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: True
BASE: /content/drive/MyDrive/driving
준비 완료


In [1]:
!nvidia-smi

Wed May 20 14:06:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

BASE = "/content/drive/MyDrive/driving"

os.makedirs(f"{BASE}/data/videos", exist_ok=True)
os.makedirs(f"{BASE}/outputs", exist_ok=True)
os.makedirs(f"{BASE}/outputs/videos", exist_ok=True)

print(f"BASE 폴더: {BASE}")
print(f"폴더 생성 완료")

BASE 폴더: /content/drive/MyDrive/driving
폴더 생성 완료


In [4]:
videos = os.listdir(f"{BASE}/data/videos")
print("영상 파일 목록:", videos)

영상 파일 목록: ['test (1).mp4']


In [5]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.1 MB/s eta 0:00:00


In [9]:
bytetrack_config = """tracker_type: bytetrack
track_high_thresh: 0.4
track_low_thresh: 0.1
new_track_thresh: 0.5
track_buffer: 90
match_thresh: 0.8
fuse_score: True
"""

with open('/content/custom_bytetrack.yaml', 'w') as f:
    f.write(bytetrack_config)

print("custom_bytetrack.yaml 생성 완료")

custom_bytetrack.yaml 생성 완료


In [2]:
from ultralytics import YOLO
import json
import time
import cv2

VIDEO_NAME = "test.mp4"

video_path = f"{BASE}/data/videos/{VIDEO_NAME}"

# 영상 정보 미리 읽기
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames_video = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(f"영상 정보: {total_frames_video}프레임, {fps} fps, 길이 {total_frames_video/fps:.1f}초")

# YOLO 모델 로드
model = YOLO("yolo11n.pt")

# 처리 시작
t0 = time.time()

results = model.track(
    source=video_path,
    tracker='/content/custom_bytetrack.yaml',
    classes=[2, 3, 5, 7],     # car, motorcycle, bus, truck
    conf=0.15,                # 검출 누락 줄이려고 낮춤
    persist=True,             # 프레임 간 ID 유지
    stream=True,              # 메모리 효율
    save=True,                # 결과 영상 저장
    project=f"{BASE}/outputs/videos",
    name="track",
    exist_ok=True,
    verbose=False
)

# 프레임 순회하면서 JSON 데이터 수집
frames_data = []

for frame_idx, r in enumerate(results):
    vehicles = []

    if r.boxes is not None and r.boxes.id is not None:
        boxes = r.boxes.xyxy.cpu().numpy()
        track_ids = r.boxes.id.cpu().numpy().astype(int)

        for box, tid in zip(boxes, track_ids):
            x1, y1, x2, y2 = box.tolist()
            pos_x = (x1 + x2) / 2
            pos_y = y2

            vehicles.append({
                "track_id": int(tid),
                "bbox_pixel": [x1, y1, x2, y2],
                "position_road_m": [pos_x, pos_y],
                "lane_id": 0,
                "lateral_offset_m": 0.0,
                "speed_est_mps": 0.0
            })

    frames_data.append({
        "frame_id": frame_idx,
        "timestamp_sec": frame_idx / fps,
        "vehicles": vehicles
    })

elapsed = time.time() - t0
video_duration = total_frames_video / fps
print(f"\n=== 처리 완료 ===")
print(f"영상 길이: {video_duration:.1f}초")
print(f"처리 시간: {elapsed:.1f}초")
print(f"속도: {video_duration/elapsed:.2f}배속")
print(f"수집된 프레임: {len(frames_data)}개")

영상 정보: 150프레임, 15.0 fps, 길이 10.0초
Results saved to /content/drive/MyDrive/driving/outputs/videos/track

=== 처리 완료 ===
영상 길이: 10.0초
처리 시간: 11.2초
속도: 0.89배속
수집된 프레임: 150개


In [4]:
output = {
    "version": "v1.0",
    "note": "1주차 산출물. position_road_m, lane_id, lateral_offset_m, speed_est_mps는 더미값. 2주차에 호모그래피+차선검출로 실제값 교체 예정.",
    "fps": fps,
    "total_frames": len(frames_data),
    "frames": frames_data
}

output_path = f"{BASE}/outputs/test_tracks_v1.json"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"저장 완료: {output_path}")

저장 완료: /content/drive/MyDrive/driving/outputs/test_tracks_v1.json


In [5]:
#결과 검증 (ID 안정성 확인)

from collections import defaultdict

with open(output_path) as f:
    data = json.load(f)

track_frames = defaultdict(list)
for frame in data['frames']:
    for v in frame['vehicles']:
        track_frames[v['track_id']].append(frame['frame_id'])

print(f"총 unique track_id 개수: {len(track_frames)}\n")
print("track_id별 등장 통계:")
for tid in sorted(track_frames.keys()):
    frames = track_frames[tid]
    duration_sec = (frames[-1] - frames[0]) / fps
    print(f"  ID {tid:3d}: {len(frames):4d}프레임, {frames[0]:4d}~{frames[-1]:4d} ({duration_sec:.1f}초)")

empty_frames = sum(1 for f in data['frames'] if len(f['vehicles']) == 0)
print(f"\n차량 0대인 프레임: {empty_frames}/{len(data['frames'])}")

총 unique track_id 개수: 12

track_id별 등장 통계:
  ID   1:    4프레임,    2~   7 (0.3초)
  ID   2:   54프레임,   23~  78 (3.7초)
  ID   3:   57프레임,   29~  85 (3.7초)
  ID   6:    8프레임,   31~  38 (0.5초)
  ID   8:   13프레임,   46~  58 (0.8초)
  ID   9:    2프레임,   59~  60 (0.1초)
  ID  11:    1프레임,   71~  71 (0.0초)
  ID  13:    1프레임,   75~  75 (0.0초)
  ID  14:    1프레임,   75~  75 (0.0초)
  ID  15:    1프레임,   75~  75 (0.0초)
  ID  16:    1프레임,   77~  77 (0.0초)
  ID  18:   16프레임,  124~ 149 (1.7초)

차량 0대인 프레임: 69/150


In [6]:
import os

schema_content = """# 출력 스키마 v1.0

A 파이프라인 → C 룰 모듈로 전달되는 JSON 데이터 구조.

## 파일 구조

루트 레벨:
- `version` (str): 스키마 버전 ("v1.0" = 1주차 더미값 포함, "v1.1" = 2주차 실제값)
- `note` (str): 비고
- `fps` (float): 원본 영상 프레임 레이트
- `total_frames` (int): 총 프레임 수
- `frames` (list): 프레임 배열

각 프레임:
- `frame_id` (int): 프레임 번호 (0부터)
- `timestamp_sec` (float): 시각 (초) = frame_id / fps
- `vehicles` (list): 그 프레임에서 검출된 차량 배열 (비어있을 수 있음)

각 차량:
- `track_id` (int): 추적 ID. 같은 차량이면 영상 내내 동일 유지(목표)
- `bbox_pixel` (list[float]): 픽셀 박스 [x1, y1, x2, y2] (좌상, 우하)
- `position_road_m` (list[float]): 도로 평면 좌표 [x, y] (미터)
- `lane_id` (int): 차선 번호 (1=가장 왼쪽 차선)
- `lateral_offset_m` (float): 차선 중심선 대비 횡방향 편차 (미터, 양수=오른쪽, 음수=왼쪽)
- `speed_est_mps` (float): 추정 속도 (m/s)

## 좌표계 정의

- **픽셀 좌표 (`bbox_pixel`)**: 영상 좌상단 원점. x는 오른쪽, y는 아래쪽 양의 방향. OpenCV 관례.
- **도로 좌표 (`position_road_m`)**: 호모그래피 변환 후의 평면 좌표 (미터). 차량 위치 기준점은 bbox 하단 중앙 = (`(x1+x2)/2`, `y2`).

## 현재 버전(v1.0) 한계 (1주차 종료 시점)

다음 필드들은 **더미값**으로 채워져 있음:
- `position_road_m`: 호모그래피 미적용. bbox 하단 중앙의 픽셀 좌표를 그대로 넣은 임시값
- `lane_id`: 차선 검출 미적용. 0으로 통일
- `lateral_offset_m`: 0.0으로 통일
- `speed_est_mps`: 0.0으로 통일

2주차 종료 시 v1.1로 교체 예정. **스키마(필드 이름/타입)는 동결, 값만 실제값으로 채워짐.**

## 룰 적용 가이드 (C 모듈 참고용)

### 차선 표류 (Lane Weaving)
- 입력: 동일 `track_id`의 `lateral_offset_m` 시계열
- 계산: 슬라이딩 윈도우(예: 10초) 표준편차 σ
- 임계 예시: σ > 0.4m가 5초 이상 지속 → 의심

### 짧은 차간거리 (Tailgating)
- 입력: 같은 `lane_id`의 두 차량의 `position_road_m`, `speed_est_mps`
- 계산: (앞차와의 도로 좌표상 거리) ÷ (자차 속도) = time headway
- 임계 예시: < 1.0초가 5초 이상 지속 → 의심

## 빈 프레임 처리

차량 미검출 프레임은 `vehicles: []`로 표기. 프레임 자체를 생략하지 않음.

## 변경 이력

- **v1.0** (1주차 종료): 초기 확정. position_road_m, lane_id, lateral_offset_m, speed_est_mps는 더미값
- **v1.1** (2주차 종료 예정): 호모그래피 + 차선 검출 적용, 실제값으로 교체

## 스키마 변경 정책

v1.0 확정 이후 필드명/타입 변경은 A에게 사전 협의 필수.
"""

os.makedirs(f"{BASE}/configs", exist_ok=True)
schema_path = f"{BASE}/configs/schema_v1.md"
with open(schema_path, "w", encoding="utf-8") as f:
    f.write(schema_content)
print(f"저장 완료: {schema_path}")

저장 완료: /content/drive/MyDrive/driving/configs/schema_v1.md


In [7]:
import os
import json
from collections import defaultdict
from datetime import datetime

# JSON 다시 읽기
with open(output_path) as f:
    data = json.load(f)

# track_id별 등장 프레임 수집
track_frames = defaultdict(list)
for frame in data['frames']:
    for v in frame['vehicles']:
        track_frames[v['track_id']].append(frame['frame_id'])

# 기본 통계
total_frames = len(data['frames'])
unique_tracks = len(track_frames)
empty_frames = sum(1 for f in data['frames'] if len(f['vehicles']) == 0)
empty_ratio = empty_frames / total_frames * 100

# 차량별 지속시간 통계
durations = []
for tid, frames in track_frames.items():
    duration_sec = (frames[-1] - frames[0]) / fps
    durations.append((tid, len(frames), frames[0], frames[-1], duration_sec))

durations.sort(key=lambda x: x[0])  # track_id 순 정렬

# 짧게 등장한 트랙 개수 (1초 미만 = ID switching 의심)
short_tracks = sum(1 for d in durations if d[4] < 1.0)
long_tracks = sum(1 for d in durations if d[4] >= 3.0)

# ===== 콘솔 출력 =====
print(f"총 unique track_id 개수: {unique_tracks}\n")
print("track_id별 등장 통계:")
for tid, n_frames, f_start, f_end, dur in durations:
    print(f"  ID {tid:3d}: {n_frames:4d}프레임, {f_start:4d}~{f_end:4d} ({dur:.1f}초)")
print(f"\n차량 0대인 프레임: {empty_frames}/{total_frames}")

# ===== 마크다운 보고서 생성 =====
lines = []
lines.append("# 1주차 추적 안정성 검증 보고서\n")
lines.append(f"**생성 일시:** {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
lines.append(f"**대상 파일:** `{output_path}`\n")
lines.append(f"**영상 FPS:** {fps}\n")
lines.append(f"**총 프레임 수:** {total_frames}\n")
lines.append("")

lines.append("## 요약\n")
lines.append(f"- **Unique track_id 개수:** {unique_tracks}")
lines.append(f"- **차량 0대 프레임:** {empty_frames} / {total_frames} ({empty_ratio:.1f}%)")
lines.append(f"- **단명 트랙 (1초 미만):** {short_tracks}개 — ID switching 의심 지표")
lines.append(f"- **장수 트랙 (3초 이상):** {long_tracks}개 — 안정적으로 추적된 차량")
lines.append("")

lines.append("## ByteTrack 튜닝 설정\n")
lines.append("`/content/custom_bytetrack.yaml`")
lines.append("```yaml")
lines.append("tracker_type: bytetrack")
lines.append("track_high_thresh: 0.4")
lines.append("track_low_thresh: 0.1")
lines.append("new_track_thresh: 0.5")
lines.append("track_buffer: 90")
lines.append("match_thresh: 0.8")
lines.append("fuse_score: True")
lines.append("```")
lines.append("")
lines.append("- `conf=0.15` (검출 신뢰도 임계)")
lines.append("- `track_buffer=90` (검출 끊김 허용 3초 @ 30fps)")
lines.append("")

lines.append("## track_id별 상세 통계\n")
lines.append("| track_id | 등장 프레임 수 | 시작 프레임 | 종료 프레임 | 지속 시간(초) |")
lines.append("|---|---|---|---|---|")
for tid, n_frames, f_start, f_end, dur in durations:
    lines.append(f"| {tid} | {n_frames} | {f_start} | {f_end} | {dur:.1f} |")
lines.append("")

lines.append("## 해석\n")
lines.append("- **Unique track_id 개수**가 영상에 실제 등장한 차량 수와 가까울수록 좋음. 크게 많으면 ID switching이 발생한 것.")
lines.append("- **단명 트랙(1초 미만)**은 ID switching이나 일시적 오검출의 흔적일 가능성. 2주차 호모그래피 적용 후 재평가.")
lines.append("- **차량 0대 프레임 비율**이 높다면 검출 누락이거나 실제로 차량이 없는 구간. 영상 직접 확인 필요.")
lines.append("")

lines.append("## 향후 계획\n")
lines.append("- 2주차: 호모그래피 + 차선 검출 적용 후 동일 영상으로 재측정")
lines.append("- 본 보고서의 unique_tracks / 단명 트랙 수를 비교 기준으로 사용")
lines.append("")

report_content = "\n".join(lines)

# 저장
os.makedirs(f"{BASE}/outputs/reports", exist_ok=True)
report_path = f"{BASE}/outputs/reports/tracking_report_v1.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_content)
print(f"\n보고서 저장 완료: {report_path}")

총 unique track_id 개수: 12

track_id별 등장 통계:
  ID   1:    4프레임,    2~   7 (0.3초)
  ID   2:   54프레임,   23~  78 (3.7초)
  ID   3:   57프레임,   29~  85 (3.7초)
  ID   6:    8프레임,   31~  38 (0.5초)
  ID   8:   13프레임,   46~  58 (0.8초)
  ID   9:    2프레임,   59~  60 (0.1초)
  ID  11:    1프레임,   71~  71 (0.0초)
  ID  13:    1프레임,   75~  75 (0.0초)
  ID  14:    1프레임,   75~  75 (0.0초)
  ID  15:    1프레임,   75~  75 (0.0초)
  ID  16:    1프레임,   77~  77 (0.0초)
  ID  18:   16프레임,  124~ 149 (1.7초)

차량 0대인 프레임: 69/150

보고서 저장 완료: /content/drive/MyDrive/driving/outputs/reports/tracking_report_v1.md
